In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

import numpy as np
import random
from copy import deepcopy
from tqdm.auto import tqdm

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [2]:
activations = {"relu": nn.ReLU(), "tanh": nn.Tanh(), "gelu": nn.GELU()}

In [3]:
class MLP(nn.Module):

    def __init__(self, input_dim, output_dim, hidden_dims, activation) :

        super().__init__()

        inp_prev = input_dim

        activation = activations[activation]

        layers = []

        for h in hidden_dims:

            layers.append(nn.Linear(inp_prev, h))
            layers.append(activation)
            inp_prev = h

        layers.append(nn.Linear(inp_prev, output_dim))

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [4]:
def train_model(model,
                train_loader,
                valid_loader,
                optimizer_name,
                lr,
                weight_decay,
                epochs=50,
                device="cpu"):

    model.to(device)

    criterion = nn.CrossEntropyLoss()

    if optimizer_name == "adam":
        optimizer = optim.Adam(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay
        )

    else:

        optimizer = optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=0.9,
            weight_decay=weight_decay
        )

    for _ in range(epochs):

        model.train()

        for x, y in train_loader:

            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            logits = model(x)

            loss = criterion(logits, y)

            loss.backward()

            optimizer.step()

    model.eval()

    total_loss = 0
    n = 0

    with torch.no_grad():

        for x, y in valid_loader:

            x = x.to(device)
            y = y.to(device)

            logits = model(x)

            loss = criterion(logits, y)

            total_loss += loss.item() * len(x)

            n += len(x)

    return total_loss / n

In [5]:
def random_state():

    n_layers = random.randint(1, 4)

    hidden_dims = [
        random.choice([32, 64, 128, 256])
        for _ in range(n_layers)
    ]

    return {
        "lr": 10**np.random.uniform(-4, -1),
        "weight_decay": 10**np.random.uniform(-6, -2),
        "optimizer": random.choice(["adam", "sgd"]),
        "activation": random.choice(
            ["relu", "tanh", "gelu"]
        ),
        "hidden_dims": hidden_dims
    }

In [6]:
def transition_kernel(state):

    new_state = deepcopy(state)

    move = random.choice([
        "lr",
        "weight_decay",
        "optimizer",
        "activation",
        "layers",
        "neurons"
    ])

    # LR
    if move == "lr":

        log_lr = np.log10(state["lr"])

        log_lr += np.random.normal(0, 0.3)

        log_lr = np.clip(log_lr, -5, -1)

        new_state["lr"] = 10**log_lr

    # L2
    elif move == "weight_decay":

        log_wd = np.log10(state["weight_decay"])

        log_wd += np.random.normal(0, 0.5)

        log_wd = np.clip(log_wd, -7, 1)

        new_state["weight_decay"] = 10**log_wd

    # optimizer
    elif move == "optimizer":

        new_state["optimizer"] = (
            "adam"
            if state["optimizer"] == "sgd"
            else "sgd"
        )

    # activation
    elif move == "activation":

        acts = ["relu", "tanh", "gelu"]

        acts.remove(state["activation"])

        new_state["activation"] = random.choice(acts)

    # numero de capas
    elif move == "layers":

        current = len(state["hidden_dims"])

        delta = random.choice([-1, 1])

        new_layers = np.clip(current + delta, 1, 5)

        hidden = deepcopy(state["hidden_dims"])

        if new_layers > current:

            hidden.append(
                random.choice([16, 32,64,128,256])
            )

        else:

            hidden = hidden[:new_layers]

        new_state["hidden_dims"] = hidden

    # neuronas
    elif move == "neurons":

        hidden = deepcopy(state["hidden_dims"])

        # elegir capa aleatoria
        idx = random.randint(
            0,
            len(hidden)-1
        )

        # movimiento multiplicativo
        factor = random.choice([
            0.5,
            2
        ])

        hidden[idx] = int(
            hidden[idx] * factor
        )

        # restringir tamaño
        hidden[idx] = int(
            np.clip(hidden[idx], 16, 512)
        )

        new_state["hidden_dims"] = hidden

    return new_state

In [7]:
def simulated_annealing(
        train_loader,
        valid_loader,
        input_dim,
        output_dim,
        iterations=50,
        T0=1.0,
        alpha=0.95,
        device="cpu"):

    # =========================
    # Estado inicial
    # =========================

    current = random_state()

    model = MLP(
        input_dim=input_dim,
        output_dim=output_dim,
        hidden_dims=current["hidden_dims"],
        activation=current["activation"]
    )

    current_loss = train_model(
        model=model,
        train_loader=train_loader,
        valid_loader=valid_loader,
        optimizer_name=current["optimizer"],
        lr=current["lr"],
        weight_decay=current["weight_decay"],
        device=device
    )

    # =========================
    # Mejor solución encontrada
    # =========================

    best_state = deepcopy(current)

    best_loss = current_loss

    # GUARDAR PESOS
    best_model_state = deepcopy(
        model.state_dict()
    )

    T = T0

    # tqdm
    pbar = tqdm(
        range(iterations),
        desc="Simulated Annealing"
    )

    for k in pbar:

        # =========================
        # Proposal
        # =========================

        proposal = transition_kernel(current)

        model = MLP(
            input_dim=input_dim,
            output_dim=output_dim,
            hidden_dims=proposal["hidden_dims"],
            activation=proposal["activation"]
        )

        proposal_loss = train_model(
            model=model,
            train_loader=train_loader,
            valid_loader=valid_loader,
            optimizer_name=proposal["optimizer"],
            lr=proposal["lr"],
            weight_decay=proposal["weight_decay"],
            device=device
        )

        # =========================
        # Acceptance step
        # =========================

        delta = proposal_loss - current_loss

        accept = False

        if delta < 0:

            accept = True

        else:

            prob = np.exp(-delta / T)

            if np.random.rand() < prob:

                accept = True

        # =========================
        # Update chain
        # =========================

        if accept:

            current = deepcopy(proposal)

            current_loss = proposal_loss

            # =========================
            # Mejor modelo global
            # =========================

            if current_loss < best_loss:

                best_loss = current_loss

                best_state = deepcopy(current)

                # GUARDAR PESOS
                best_model_state = deepcopy(
                    model.state_dict()
                )

        # =========================
        # Cooling
        # =========================

        T *= alpha

        # =========================
        # tqdm info
        # =========================

        pbar.set_postfix({

            "loss": f"{current_loss:.4f}",

            "best": f"{best_loss:.4f}",

            "T": f"{T:.4f}",

            "lr": f"{current['lr']:.2e}",

            "wd": f"{current['weight_decay']:.2e}",

            "opt": current["optimizer"],

            "act": current["activation"],

            "layers": len(current["hidden_dims"]),

            "hidden": str(current["hidden_dims"]),

            "accept": int(accept)

        })

    # =========================
    # Reconstruir mejor modelo
    # =========================

    best_model = MLP(
        input_dim=input_dim,
        output_dim=output_dim,
        hidden_dims=best_state["hidden_dims"],
        activation=best_state["activation"]
    )

    best_model.load_state_dict(
        best_model_state
    )

    # =========================
    # Return
    # =========================

    return best_model, best_state, best_loss

In [8]:
wine = load_wine()

X = wine.data
y = wine.target

scaler = StandardScaler()

X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


X_train = torch.tensor(
    X_train,
    dtype=torch.float32
)

y_train = torch.tensor(
    y_train,
    dtype=torch.long
)

X_test = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_test = torch.tensor(
    y_test,
    dtype=torch.long
)


train_dataset = TensorDataset(
    X_train,
    y_train
)

test_dataset = TensorDataset(
    X_test,
    y_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)


In [9]:
input_dim = X_train.shape[1]
output_dim = len(torch.unique(y_train))


In [10]:
best_model, best_hparams, best_loss = simulated_annealing(
    train_loader=train_loader,
    valid_loader=test_loader,
    input_dim=input_dim,
    output_dim=output_dim,
    iterations=100,
    T0=1.0,
    alpha=0.95,
    device="cuda"
)

Simulated Annealing:   0%|          | 0/100 [00:00<?, ?it/s]

In [12]:
best_hparams

{'lr': np.float64(0.050668996220891195),
 'weight_decay': np.float64(0.0014723549581795849),
 'optimizer': 'adam',
 'activation': 'tanh',
 'hidden_dims': [16, 512, 128, 128, 256]}

In [15]:
best_loss

0.0015071642653007682

In [16]:
best_model

MLP(
  (net): Sequential(
    (0): Linear(in_features=13, out_features=16, bias=True)
    (1): Tanh()
    (2): Linear(in_features=16, out_features=512, bias=True)
    (3): Tanh()
    (4): Linear(in_features=512, out_features=128, bias=True)
    (5): Tanh()
    (6): Linear(in_features=128, out_features=128, bias=True)
    (7): Tanh()
    (8): Linear(in_features=128, out_features=256, bias=True)
    (9): Tanh()
    (10): Linear(in_features=256, out_features=3, bias=True)
  )
)

In [20]:
y_pred = torch.softmax(best_model.forward(X_train), dim=-1).argmax(dim=-1)
(y_pred == y_train).float().mean()

tensor(1.)

In [14]:
y_train

tensor([0, 0, 0, 0, 2, 2, 1, 2, 0, 0, 1, 1, 0, 0, 2, 1, 1, 1, 0, 0, 1, 2, 0, 2,
        2, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 2, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0,
        2, 0, 0, 2, 1, 1, 2, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 2, 2,
        0, 0, 2, 1, 1, 1, 1, 2, 0, 2, 0, 2, 1, 2, 2, 1, 0, 0, 1, 1, 1, 0, 0, 0,
        2, 1, 1, 1, 2, 1, 0, 1, 0, 1, 2, 0, 0, 1, 2, 2, 1, 1, 1, 2, 2, 2, 2, 1,
        1, 2, 1, 2, 2, 0, 0, 2, 0, 1, 2, 2, 1, 2, 1, 1, 1, 2, 1, 0, 2, 2])